# 🧠 Tactile SDF Reconstruction — Google Colab

This notebook trains a **PointNet + SIREN** model to reconstruct 3D shapes from sparse tactile contact points.

**Data Source:** Hugging Face Hub (`huggingface_hub`)
**Architecture:** PointNet Encoder + SIREN Decoder

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install trimesh rtree plotly streamlit scikit-image scikit-learn tqdm huggingface_hub
!pip install fast-simplification  # For faster mesh decimation

In [ ]:
# @title 📂 Step 2: Clone Repo & Download Data
import os
from huggingface_hub import snapshot_download

# 1. Clone the tactile-sdf code
if not os.path.exists('tactile-sdf'):
    !git clone https://github.com/635jack/tactile-sdf.git

# 2. Download the dataset from Hugging Face
# Replace with your actual repo ID (e.g., '635jack/grasp-dataset-curated')
repo_id = "635jack/grasp-dataset-curated" 

print(f"📥 Downloading dataset from {repo_id}...")
dataset_path = snapshot_download(
    repo_id=repo_id, 
    repo_type="dataset",
    local_dir="grasp-dataset-curated"
)

%cd tactile-sdf

> [!IMPORTANT]
> **Mesh Path**: If your Hugging Face dataset contains both the meshes (`.glb`) and the grasp data (`.npz`), ensure the paths in the commands below match your `grasp-dataset-curated` structure.

In [ ]:
# @title 📐 Step 3: Run SDF Preprocessing
# Adjust --glb_dir and --dataset_dir to point to your downloaded HF folder
!python preprocess_sdf.py \
    --n_points 50000 \
    --glb_dir ../grasp-dataset-curated/data/objaverse \
    --dataset_dir ../grasp-dataset-curated

In [ ]:
# @title 🚀 Step 4: Start Training
!python train.py \
    --epochs 300 \
    --batch_size 8 \
    --dataset_dir ../grasp-dataset-curated

In [ ]:
# @title 📊 Step 5: Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

runs = sorted(glob.glob('runs/*'))
if runs:
    latest_run = runs[-1]
    img_path = os.path.join(latest_run, 'training_curves.png')
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No training runs found.")